In [38]:
import tbparse as tb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

In [39]:
lw = 3
SMALL_SIZE = 10
MEDIUM_SIZE = 12
BIGGER_SIZE = 14

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [63]:
DATASETS = {
    "ignition_subsample": "ignition",
    "ignition_subsample_reduced_4": "ignition",
    "ignition_subsample_no_compand": "ignition",
    "ignition_fjlt": "ignition",
    "ignition_fjlt_no_compand": "ignition",
    "neuron_subsample": "neuron_transport",
    "neuron_subsample_reduced_4": "neuron_transport",
    "neuron_subsample_no_compand": "neuron_transport",
    "neuron_fjlt": "neuron_transport",
    "neuron_fjlt_no_compand": "neuron_transport",
    "channel_subsample": "channel_flow",
    "channel_subsample_reduced_4": "channel_flow",
    "channel_subsample_no_compand": "channel_flow",
    "channel_fjlt": "channel_flow",
    "channel_fjlt_no_compand": "channel_flow",
}

METHOD_DIRS = {
    "ignition_subsample": "hnet_online_subsample",
    "ignition_subsample_reduced_4": "hnet_online_subsample_reduced_4",
    "ignition_subsample_no_compand": "hnet_online_subsample_no_compand",
    "ignition_fjlt": "hnet_online_fjlt",
    "ignition_fjlt_no_compand": "hnet_online_fjlt_no_compand",
    "neuron_subsample": "hnet_online_subsample",
    "neuron_subsample_reduced_4": "hnet_online_subsample_reduced_4",
    "neuron_subsample_no_compand": "hnet_online_subsample_no_compand",
    "neuron_fjlt": "hnet_online_fjlt",
    "neuron_fjlt_no_compand": "hnet_online_fjlt_no_compand",
    "channel_subsample": "hnet_online_subsample",
    "channel_subsample_reduced_4": "hnet_online_subsample_reduced_4",
    "channel_subsample_no_compand": "hnet_online_subsample_no_compand",
    "channel_fjlt": "hnet_online_fjlt",
    "channel_fjlt_no_compand": "hnet_online_fjlt_no_compand",
}

METRICS = {
    "PSNR": "test_avg_PSNR",
    "MSE": "test_avg_MSE",
    "Compression Factor": "compression_factor"
}


LOG_GROUP_DIRS = {
    "ignition": "alex_ignition",
    "neuron_transport": "alex_neuron_transport",
    "channel_flow": "alex_channel_flow",
}


def latest_version_dir(method_dir):
    version_dirs = [
        path
        for path in method_dir.glob("version_*")
        if path.is_dir()
        and path.name.split("version_", 1)[-1].isdigit()
    ]

    if not version_dirs:
        raise FileNotFoundError(
            f"No version directories found in {method_dir}"
        )

    return max(
        version_dirs,
        key=lambda path: int(path.name.split("_")[-1]),
    )


def get_version_dir(run_name, dataset):
    method_dir = (
        Path("lightning_logs")
        / dataset
        / LOG_GROUP_DIRS[dataset]
        / METHOD_DIRS[run_name]
    )
    return latest_version_dir(method_dir)

Q_STRATEGIES = {
    "real_": "FP32",
    "none_8_": "INT8",
    "fwht_8_": "FWHT + INT8",
}


def metric_strategy_prefixes(method):
    """Return valid TensorBoard prefixes, newest first."""
    if method.endswith("_reduced_4"):
        return {
            "FP32": ("real_",),
        }

    if method.endswith("_no_compand"):
        return {
            "FP32": ("real_",),
            "INT8": ("none_8_no_compand_", "none_8_"),
            "FWHT + INT8": ("fwht_8_no_compand_", "fwht_8_"),
        }

    return {
        "FP32": ("real_",),
        "INT8": ("none_8_compand_", "none_8_"),
        "FWHT + INT8": ("fwht_8_compand_", "fwht_8_"),
    }

In [65]:
# new pandas DataFrame to hold the metrics for each strategy,
results_df = pd.DataFrame(columns=["Dataset", "Method", "Version", "Strategy"] + list(METRICS.values()))
print(results_df)

# for each strategy, extract the metrics and construct a DataFrame
for run_name, dataset in DATASETS.items():
    version_dir = get_version_dir(run_name, dataset)

    scalars = tb.SummaryReader(str(version_dir)).scalars
    report_scalars = scalars[
        scalars["tag"].str.contains("test_avg|compression_factor")
    ][["tag", "value"]]

    method = METHOD_DIRS[run_name]
    strategies = metric_strategy_prefixes(method)

    for strategy_label, possible_prefixes in strategies.items():
        prefix = next(
            (
                candidate
                for candidate in possible_prefixes
                if report_scalars["tag"].str.startswith(candidate).any()
            ),
            possible_prefixes[0],
        )
        strategy_scalars = report_scalars[
            report_scalars["tag"].str.startswith(prefix)
        ]

        metrics_dict = {}

        # each metric is one column in the DataFrame, with the strategy as the row, and the value as the cell
        for metric_name, metric in METRICS.items():

            metric_tag = f"{prefix}{metric}"
            metric_value = strategy_scalars[
                strategy_scalars["tag"] == metric_tag
            ]["value"]

            if metric_value.empty:
                metric_value = None
            else:
                metric_value = float(metric_value.iloc[-1])
            metrics_dict[metric] = metric_value

        print(f"Metrics for {dataset} with {strategy_label}: {metrics_dict}")

        results_df = pd.concat(
            [
                results_df,
                pd.DataFrame(
                    {
                        "Dataset": dataset,
                        "Method": METHOD_DIRS[run_name],
                        "Version": version_dir.name,
                        "Strategy": strategy_label,
                        **metrics_dict,
                    },
                    index=[0],
                ),
            ],
            ignore_index=True,
        )
results_df;

Empty DataFrame
Columns: [Dataset, Method, Version, Strategy, test_avg_PSNR, test_avg_MSE, compression_factor]
Index: []
Metrics for neuron_transport with FP32: {'test_avg_PSNR': 30.228994369506836, 'test_avg_MSE': 9398.166015625, 'compression_factor': 1882.7156982421875}
Metrics for neuron_transport with INT8: {'test_avg_PSNR': 18.74677848815918, 'test_avg_MSE': 150471.46875, 'compression_factor': 7529.892578125}
Metrics for neuron_transport with Hadamard + INT8: {'test_avg_PSNR': 29.790142059326172, 'test_avg_MSE': 10342.0283203125, 'compression_factor': 7136.76318359375}


,Dataset,Method,Version,Strategy,test_avg_PSNR,test_avg_MSE,compression_factor
0,neuron_transport,hnet_online_subsample,version_7,FP32,30.228994,9398.166016,1882.715698
1,neuron_transport,hnet_online_subsample,version_7,INT8,18.746778,150471.46875,7529.892578
2,neuron_transport,hnet_online_subsample,version_7,Hadamard + INT8,29.790142,10342.02832,7136.763184


## Quantization comparison tables

The full `results_df` is retained as the source data, while the tables below select only the runs and quantization strategies relevant to each comparison.


In [ ]:
METRIC_COLUMNS = list(METRICS.values())
DATASET_ORDER = list(dict.fromkeys(DATASETS.values()))


def make_comparison_table(selections):
    """Build a table from (method, strategy, display label) selections."""
    selected_rows = []

    for display_order, (method, strategy, model_label) in enumerate(selections):
        rows = results_df[
            (results_df["Method"] == method)
            & (results_df["Strategy"] == strategy)
        ].copy()
        rows = rows.dropna(subset=METRIC_COLUMNS, how="all")

        if rows.empty:
            print(f"Warning: no results for {method}, {strategy}")
            continue

        rows["Model"] = model_label
        rows["_display_order"] = display_order
        selected_rows.append(rows)

    if not selected_rows:
        return pd.DataFrame(
            columns=["Dataset", "Model", "PSNR", "MSE", "Compression Factor"]
        )

    table = pd.concat(selected_rows, ignore_index=True)
    table["Dataset"] = pd.Categorical(
        table["Dataset"], categories=DATASET_ORDER, ordered=True
    )
    table = table.sort_values(["Dataset", "_display_order"])

    return (
        table[
            ["Dataset", "Model", "test_avg_PSNR", "test_avg_MSE", "compression_factor"]
        ]
        .rename(
            columns={
                "test_avg_PSNR": "PSNR",
                "test_avg_MSE": "MSE",
                "compression_factor": "Compression Factor",
            }
        )
        .reset_index(drop=True)
    )


### INT8 and FWHT + INT8 versus reduced FP32


In [ ]:
quantized_vs_reduced = make_comparison_table([
    ("hnet_online_subsample", "INT8", "INT8 (companded)"),
    ("hnet_online_subsample", "FWHT + INT8", "FWHT + INT8 (companded)"),
    ("hnet_online_subsample_reduced_4", "FP32", "Reduced FP32"),
])
quantized_vs_reduced


### FP32 versus INT8 and FWHT + INT8


In [ ]:
fp32_vs_quantized = make_comparison_table([
    ("hnet_online_subsample", "FP32", "FP32"),
    ("hnet_online_subsample", "INT8", "INT8 (companded)"),
    ("hnet_online_subsample", "FWHT + INT8", "FWHT + INT8 (companded)"),
])
fp32_vs_quantized


### FWHT + INT8 with and without companding


In [ ]:
companding_comparison = make_comparison_table([
    ("hnet_online_subsample", "FWHT + INT8", "FWHT + INT8 with companding"),
    ("hnet_online_subsample_no_compand", "FWHT + INT8", "FWHT + INT8 without companding"),
])
companding_comparison


### FP32 versus companded FWHT + INT8


In [ ]:
fp32_vs_fwht = make_comparison_table([
    ("hnet_online_subsample", "FP32", "FP32"),
    ("hnet_online_subsample", "FWHT + INT8", "FWHT + INT8 (companded)"),
])
fp32_vs_fwht


## FJLT quantization comparison tables


### FJLT INT8 and FWHT + INT8 versus reduced FP32

No FJLT reduced model was run, so this comparison uses the reduced subsampling model as the FP32 size-matched baseline.


In [ ]:
fjlt_quantized_vs_reduced = make_comparison_table([
    ("hnet_online_fjlt", "INT8", "FJLT INT8 (companded)"),
    ("hnet_online_fjlt", "FWHT + INT8", "FJLT FWHT + INT8 (companded)"),
    ("hnet_online_subsample_reduced_4", "FP32", "Reduced FP32 (subsampling)"),
])
fjlt_quantized_vs_reduced


### FJLT FP32 versus INT8 and FWHT + INT8


In [ ]:
fjlt_fp32_vs_quantized = make_comparison_table([
    ("hnet_online_fjlt", "FP32", "FJLT FP32"),
    ("hnet_online_fjlt", "INT8", "FJLT INT8 (companded)"),
    ("hnet_online_fjlt", "FWHT + INT8", "FJLT FWHT + INT8 (companded)"),
])
fjlt_fp32_vs_quantized


### FJLT FWHT + INT8 with and without companding


In [ ]:
fjlt_companding_comparison = make_comparison_table([
    ("hnet_online_fjlt", "FWHT + INT8", "FJLT FWHT + INT8 with companding"),
    ("hnet_online_fjlt_no_compand", "FWHT + INT8", "FJLT FWHT + INT8 without companding"),
])
fjlt_companding_comparison


### FJLT FP32 versus companded FWHT + INT8


In [ ]:
fjlt_fp32_vs_fwht = make_comparison_table([
    ("hnet_online_fjlt", "FP32", "FJLT FP32"),
    ("hnet_online_fjlt", "FWHT + INT8", "FJLT FWHT + INT8 (companded)"),
])
fjlt_fp32_vs_fwht


## Reconstruction comparison

This plot uses the runs selected in `DATASETS` and `METHOD_DIRS`, automatically chooses each run's latest version, and plots the reconstructions selected in `Q_STRATEGIES`. Each inset shows absolute reconstruction error on a shared logarithmic scale. Before running this section, set `misc.export: true` in each selected version's `config.yaml` and rerun testing to create the `*_reconstruction.npy` files.


In [66]:
# Plot code from ChatGPT, modified from Paper Results.ipynb to plot reconstruction.npy and with error insets.

import numpy as np
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LogNorm, Normalize
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Choose the frame, channel, and view used for each dataset.
PLOT_SETTINGS = {
    "ignition": {
        "kind": "grid",
        "frame": 12,
        "channel": 1,
        "shape": (50, 50),
    },
    "neuron_transport": {
        "kind": "scatter",
        "frame": 250,
        "channel": 0,
        "projection": (0, 1),
        "stride": 4,
        "point_size": 0.6,
    },
    "channel_flow": {
        "kind": "volume_slice",
        "frame": 249,
        "channel": 0,
        "shape": (64, 64, 64),
        "slice_axis": 2,
        "slice_index": 31,
    },
}


def select_plot_field(values, settings):
    """Extract the configured 2D view, or point values, from a dataset."""
    field = values[settings["frame"], :, settings["channel"]]

    if settings["kind"] == "grid":
        return np.asarray(field).reshape(settings["shape"])

    if settings["kind"] == "volume_slice":
        volume = np.asarray(field).reshape(settings["shape"])
        return np.take(
            volume,
            settings["slice_index"],
            axis=settings["slice_axis"],
        )

    if settings["kind"] == "scatter":
        return np.asarray(field)

    raise ValueError(f"Unknown plot kind: {settings['kind']}")


def draw_field(ax, field, dataset, settings, *, cmap, norm):
    """Draw either a regular image or an irregular point projection."""
    if settings["kind"] == "scatter":
        points = np.load(Path("data") / dataset / "points.npy", mmap_mode="r")
        x_index, y_index = settings["projection"]
        stride = settings.get("stride", 1)
        selected = slice(None, None, stride)
        artist = ax.scatter(
            points[selected, x_index],
            points[selected, y_index],
            c=field[selected],
            s=settings.get("point_size", 1.0),
            cmap=cmap,
            norm=norm,
            linewidths=0,
            rasterized=True,
        )
        ax.set_aspect("equal")
        return artist

    return ax.imshow(field, cmap=cmap, norm=norm, origin="lower")


# Load the original data and every requested reconstruction.
plot_rows = []
all_errors = []

for run_name, dataset in DATASETS.items():
    if dataset not in PLOT_SETTINGS:
        raise KeyError(f"Add plot settings for dataset {dataset!r}")

    settings = PLOT_SETTINGS[dataset]
    version_dir = get_version_dir(run_name, dataset)

    original_path = Path("data") / dataset / "features.npy"
    original_values = np.load(original_path, mmap_mode="r")
    original_field = select_plot_field(original_values, settings)

    reconstruction_fields = {}
    for prefix, strategy_label in Q_STRATEGIES.items():
        reconstruction_path = version_dir / f"{prefix}reconstruction.npy"
        if not reconstruction_path.exists():
            raise FileNotFoundError(
                f"Missing {reconstruction_path}. Set misc.export: true in "
                f"{version_dir / 'config.yaml'} and rerun testing."
            )

        reconstruction_values = np.load(reconstruction_path, mmap_mode="r")
        reconstruction_field = select_plot_field(reconstruction_values, settings)
        reconstruction_fields[strategy_label] = reconstruction_field
        all_errors.append(np.abs(original_field - reconstruction_field))

    plot_rows.append({
        "dataset": dataset,
        "settings": settings,
        "original": original_field,
        "reconstructions": reconstruction_fields,
    })


# All error insets use the same logarithmic normalization.
positive_errors = [error[error > 0] for error in all_errors if np.any(error > 0)]
if positive_errors:
    error_vmin = min(float(values.min()) for values in positive_errors)
    error_vmax = max(float(error.max()) for error in all_errors)
    error_vmax = max(error_vmax, error_vmin * 10)
else:
    error_vmin, error_vmax = 1e-12, 1.0

error_norm = LogNorm(vmin=error_vmin, vmax=error_vmax)

column_titles = ["Original", *Q_STRATEGIES.values()]
fig, axes = plt.subplots(
    len(plot_rows),
    len(column_titles),
    figsize=(4 * len(column_titles), 4 * len(plot_rows)),
    squeeze=False,
    constrained_layout=True,
)

for row_index, row in enumerate(plot_rows):
    dataset = row["dataset"]
    settings = row["settings"]
    original_field = row["original"]
    reconstructed_fields = list(row["reconstructions"].values())

    # Use a common value scale across the original and reconstructions in this row.
    row_fields = [original_field, *reconstructed_fields]
    value_vmin = min(float(np.nanmin(field)) for field in row_fields)
    value_vmax = max(float(np.nanmax(field)) for field in row_fields)
    value_norm = Normalize(vmin=value_vmin, vmax=value_vmax)

    original_ax = axes[row_index, 0]
    draw_field(
        original_ax, original_field, dataset, settings,
        cmap="viridis", norm=value_norm,
    )
    original_ax.set_title(
        f"{dataset.replace('_', ' ').title()}\nOriginal"
    )

    for column_index, (strategy_label, reconstructed_field) in enumerate(
        row["reconstructions"].items(), start=1
    ):
        ax = axes[row_index, column_index]
        draw_field(
            ax, reconstructed_field, dataset, settings,
            cmap="viridis", norm=value_norm,
        )
        ax.set_title(strategy_label)

        absolute_error = np.abs(original_field - reconstructed_field)
        inset_ax = inset_axes(
            ax, width="40%", height="40%",
            loc="upper right", borderpad=0.5,
        )
        draw_field(
            inset_ax, absolute_error, dataset, settings,
            cmap="magma", norm=error_norm,
        )
        inset_ax.set_title("Error", fontsize=8)
        inset_ax.grid(False)
        inset_ax.axis("off")

for ax in axes.ravel():
    ax.grid(False)
    ax.axis("off")

error_mappable = ScalarMappable(norm=error_norm, cmap="magma")
error_mappable.set_array([])
error_colorbar = fig.colorbar(
    error_mappable,
    ax=axes.ravel().tolist(),
    orientation="horizontal",
    fraction=0.04,
    pad=0.04,
)
error_colorbar.set_label("Absolute Error (log scale)")
plt.show()


FileNotFoundError: Missing lightning_logs/neuron_transport/hnet_online_subsample/version_7/real_reconstruction.npy. Set misc.export: true in lightning_logs/neuron_transport/hnet_online_subsample/version_7/config.yaml and rerun testing.